# #
# os.environ["CUDA_VISIBLE_DEVICES"] = "2" # 1,2 사용하기ㅡ


# multi_model_lwm1 v2
<br>모델 크기 차이에 따른 성능 차이 비교
<br> element_length=64, d_model=128
<br> in_proj: F_in(2048) → 512 → 256 → 64
<br> 예측 범위를 16->1에서 16->4로 변경

# Import

In [1]:
print("Hello from the multi-modal LWM1 notebook!")

Hello from the multi-modal LWM1 notebook!


In [2]:
import os
import sys

import numpy as  np
from lwm_multi_model1 import multi_modal_lwm  # 클래스 import
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Subset
import torch.nn as nn   
import time

from PIL import Image
import numpy as np

from deepverse import ParameterManager
from deepverse.scenario import ScenarioManager
from deepverse import Dataset

from deepverse.visualizers import ImageVisualizer, LidarVisualizer

# Setting 
### CUDA 몇번 사용하는지 잘 확인하기

In [3]:
# Scenes 1000
## Subcarriers 64

scenarios_name = "DT31"
config_path = f"scenarios/{scenarios_name}/param/config.m"
param_manager = ParameterManager(config_path)

params = param_manager.get_params()

param_manager.params["scenes"] =list(range(100))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")


현재 사용 중인 장치: cuda:0


# Generate a dataset

In [4]:
# scenes / subcarriers는 그대로
param_manager.params["radar"]["enable"] = False

# lidar가 dict로 존재하면 disable
if isinstance(param_manager.params.get("lidar", None), dict):
    param_manager.params["lidar"]["enable"] = False

# position이 dict로 존재하면 disable
if isinstance(param_manager.params.get("position", None), dict):
    param_manager.params["position"]["enable"] = False

# Radar, LiDAR, Position 사용하지 않으므로 False 
dataset = Dataset(param_manager)

Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (0.87s)


# Prerocessing

#### Channel

In [5]:
def get_coeffs_from_frame(frame, ue_idx=0):
    ue_obj = frame["ue"]

    # 케이스1) list/tuple이면 ue_idx로 선택
    if isinstance(ue_obj, (list, tuple)):
        ch_obj = ue_obj[ue_idx]
    else:
        # 케이스2) 단일 OFDMChannel이면 그대로 사용
        ch_obj = ue_obj

    # coeffs는 dict key가 아니라 attribute일 확률이 매우 큼
    if hasattr(ch_obj, "coeffs"):
        return ch_obj.coeffs

    # 혹시 dict라면 마지막 보험
    if isinstance(ch_obj, dict) and "coeffs" in ch_obj:
        return ch_obj["coeffs"]

    raise TypeError(f"Cannot get coeffs. ue type={type(ue_obj)}, ch type={type(ch_obj)}")


In [6]:
def get_train_min_max_realimag(frames, train_idx, us_idx=0):

    rmin, rmax =  float('inf'), float('-inf')
    imin, imax =  float('inf'), float('-inf')

    print("Calculating min/max over training set...")

    for t  in train_idx:
        frame  = frames[t]
        cooeffs  = get_coeffs_from_frame(frame, us_idx)  # (N_subcarriers, )

        rmin = min(rmin, float(cooeffs.real.min()))
        rmax = max(rmax, float(cooeffs.real.max()))
        imin = min(imin, float(cooeffs.imag.min()))
        imax = max(imax, float(cooeffs.imag.max()))

    print(f"Done. rmin={rmin}, rmax={rmax}, imin={imin}, imax={imax}")
    return (rmin, rmax), (imin, imax)

In [7]:
def preprocess_channel_coeffs_minmax(coeffs_np, r_min, r_max, i_min, i_max, device=device, eps=1e-16):
    # Convert Numpy to Tensor
    coeffs = torch.from_numpy(coeffs_np).to(torch.complex64)
    
    
    r = coeffs.real
    i = coeffs.imag
    
    # Min-Max Scaling [0, 1]
    # Add eps to denominator to prevent division by zero
    r_scaled = (r - r_min) / max(r_max - r_min, eps)
    i_scaled = (i - i_min) / max(i_max - i_min, eps)
    
    # Concat (Maintains shape like (..., 2*subcarriers))
    H = torch.cat([r_scaled, i_scaled], dim=-1).to(dtype=torch.float32)
    return H

#### image

In [8]:
IMG_SIZE = 224
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1,3,1,1)

def preprocess_img(path, img_size=IMG_SIZE):
    img = Image.open(path).convert("RGB")
    arr = np.array(img)

    x = torch.from_numpy(arr).permute(2, 0, 1).contiguous().float()
    x = x / 255.0
    x = x.unsqueeze(0)

    x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)

    # mean/std는 전역으로 빼는 게 성능상 좋음 (아래 참고)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x.squeeze(0)  # (3,224,224) CPU


# Dataset 구현

In [9]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

class MultiModalNextStepDatasetGPU(TorchDataset):
    def __init__(self, comm_frames, cam_files, ue_idx=0, past_len=16, pre_len=4, device=device,
                 r_min=0.0, r_max=1.0, i_min=0.0, i_max=1.0):

        self.comm_frames = comm_frames
        self.cam_files = list(cam_files)
        self.ue_idx = ue_idx
        self.past_len = past_len
        self.pre_len  = pre_len
        self.device = device

        self.r_min, self.r_max = r_min, r_max
        self.i_min, self.i_max = i_min, i_max

        self.N = min(len(self.comm_frames), len(self.cam_files))
        self.valid_start = past_len - 1
        # ✅ t+pre_len 까지 필요하므로
        self.valid_end = self.N - self.pre_len - 1

    def __len__(self):
        return self.valid_end - self.valid_start + 1

    def __getitem__(self, idx):
        t = self.valid_start + idx

        # 1) Image Past: (past_len,3,224,224)
        img_list = []
        for k in range(t - self.past_len + 1, t + 1):
            img_k = preprocess_img(self.cam_files[k])   # (3,224,224)
            img_list.append(img_k)
        img = torch.stack(img_list, dim=0)             # (T,3,224,224)

        # 2) Channel Past: (past_len,2048)
        ch_list = []
        for k in range(t - self.past_len + 1, t + 1):
            coeffs_np = get_coeffs_from_frame(self.comm_frames[k], ue_idx=self.ue_idx)
            h = preprocess_channel_coeffs_minmax(
                coeffs_np, self.r_min, self.r_max, self.i_min, self.i_max
            ).reshape(-1)                               # (2048,)
            ch_list.append(h)
        channel_past = torch.stack(ch_list, dim=0)     # (T,2048)

        # ✅ 3) Target Next pre_len: (pre_len,2048)
        tgt_list = []
        for j in range(1, self.pre_len + 1):
            coeffs_np_next = get_coeffs_from_frame(self.comm_frames[t + j], ue_idx=self.ue_idx)
            hj = preprocess_channel_coeffs_minmax(
                coeffs_np_next, self.r_min, self.r_max, self.i_min, self.i_max
            ).reshape(-1)
            tgt_list.append(hj)

        target = torch.stack(tgt_list, dim=0)          # (pre_len,2048)

        return channel_past, img, target

# DataLoader 구현

In [10]:
comm_frames = flatten_comm_frames(dataset.comm_dataset)
sensor = dataset.camera_dataset.sensors["unit1_cam1"]

ds = MultiModalNextStepDatasetGPU(
    comm_frames=comm_frames,
    cam_files=sensor.files,
    ue_idx=0,
    past_len=16,
    pre_len=4,     # ✅ 추가
    device=device
)

loader = DataLoader(
    ds,
    batch_size=8,
    shuffle=True,
    num_workers=8,     
    pin_memory=True   
)

ch, img, y = next(iter(loader))
print(ch.shape, img.shape, y.shape)
print(ch.device, img.device, y.device)


torch.Size([8, 16, 2048]) torch.Size([8, 16, 3, 224, 224]) torch.Size([8, 4, 2048])
cpu cpu cpu


# Fine-tuning

In [11]:
class FinetuneChannelPredictor(nn.Module):
    def __init__(self, backbone, F_in, F_out, out_steps=4,
                 pool="last", freeze_image=False, freeze_backbone=False,
                 element_length=64, d_model=128):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.out_steps = out_steps
        self.F_out = F_out

        self.in_proj = nn.Sequential(
            nn.Linear(F_in, 512), nn.GELU(),
            nn.Linear(512, 256),  nn.GELU(),
            nn.Linear(256, element_length)
        )

        # ✅ (B,D) -> (B, out_steps*F_out)
        self.head = nn.Linear(d_model, out_steps * F_out)

        if freeze_image:
            for p in self.backbone.image_embedding.parameters():
                p.requires_grad = False

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
            for p in self.in_proj.parameters():
                p.requires_grad = True
            for p in self.head.parameters():
                p.requires_grad = True

    def forward(self, ch, img):
        B = ch.size(0)

        ch = self.in_proj(ch)          # (B,T,64)
        tokens = self.backbone(ch, img)

        if self.pool == "last":
            z = tokens[:, -1, :]
        elif self.pool == "mean":
            z = tokens.mean(dim=1)
        else:
            raise ValueError(f"Unknown pool={self.pool}")

        yhat_flat = self.head(z)                       # (B, out_steps*F_out)
        yhat = yhat_flat.view(B, self.out_steps, self.F_out)  # ✅ (B,4,2048)
        return yhat

# NMSE(dB)

In [12]:
@torch.no_grad()
def nmse_db(yhat: torch.Tensor, y: torch.Tensor, eps: float = 1e-16) -> torch.Tensor:
    # yhat, y: (B, pre_len, F_step)
    num = torch.sum((yhat - y) ** 2).clamp_min(eps)  # batch 전체 합
    den = torch.sum(y ** 2).clamp_min(eps)
    nmse = num / den
    return 10.0 * torch.log10(nmse)

# Train/val Split

In [13]:
def used_frames_for_ds_indices(ds, ds_indices):
    used = set()
    T = ds.past_len
    P = ds.pre_len
    for i in ds_indices:
        t = ds.valid_start + i
        used.update(range(t - T + 1, t + P + 1))  # [t-15 .. t+4]
    return used

In [14]:
n_total = len(ds)
T = ds.past_len          # past_len = 16
P = ds.pre_len           # pre_len  = 4   (✅ 16->4이면 반드시 존재)

# ---- 1) choose k so that train=3k, val=k and NO frame-overlap is possible ----
# 한 샘플이 사용하는 comm frame 범위: [t-T+1 .. t+P]  => 길이 (T+P)
# non-overlap 3:1을 만들려면 안전하게 (T+P-1)만큼 여유가 필요
k_max = (n_total - (T + P - 1)) // 4
if k_max <= 0:
    raise ValueError(
        f"Not enough data for strict non-overlap 3:1. "
        f"n_total={n_total}, T={T}, P={P}. "
        f"Try increasing scenes or decreasing past_len/pre_len."
    )

k = k_max
n_val = k
n_train = 3 * k

train_idx = list(range(0, n_train))
val_idx   = list(range(n_total - n_val, n_total))

print("n_total:", n_total, "T:", T, "P:", P)
print("train samples:", len(train_idx), "val samples:", len(val_idx),
      "ratio:", len(train_idx)/len(val_idx))

# ---- 2) leakage check helper (✅ 16->4용) ----
def used_frames_for_ds_indices(ds, ds_indices):
    used = set()
    T = ds.past_len
    P = ds.pre_len
    for i in ds_indices:
        t = ds.valid_start + i
        used.update(range(t - T + 1, t + P + 1))  # ✅ [t-15 .. t+4]
    return used

# ---- 3) min/max ONLY from TRAIN, but using ALL frames used by TRAIN samples ----
train_used_frames = sorted(used_frames_for_ds_indices(ds, train_idx))  # ✅ 핵심 변경

(real_min, real_max), (imag_min, imag_max) = get_train_min_max_realimag(
    comm_frames, train_used_frames, us_idx=0
)

ds.r_min = real_min
ds.r_max = real_max
ds.i_min = imag_min
ds.i_max = imag_max

print("Dataset statistical values set in the dataset (from all train-used frames).")

# ---- 4) loaders ----
train_ds = Subset(ds, train_idx)
val_ds   = Subset(ds, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=8)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=8)

# ---- 5) strict leakage check ----
overlap = used_frames_for_ds_indices(ds, train_idx).intersection(
          used_frames_for_ds_indices(ds, val_idx))
print("frame-overlap count:", len(overlap))
assert len(overlap) == 0, "Leakage detected: train/val share frames!"

# ---- 6) Verify shapes ----
ch, img, y = next(iter(train_loader))
assert y.ndim == 3, f"Expected y as (B,P,F), got {y.shape}"  # ✅ 16->4는 3D

print("\n=== Data Check ===")
print("ch :", ch.shape,  " -> (B,T,F_in)")
print("img:", img.shape, " -> (B,T,3,224,224) or (B,3,224,224) depending on your ds")
print("y  :", y.shape,   " -> (B,P,F_out)")
print(f"y stats | min: {y.min().item():.4f}, max: {y.max().item():.4f}")
print("If scaling worked correctly, values should be within [0, 1].")

n_total: 81 T: 16 P: 4
train samples: 45 val samples: 15 ratio: 3.0
Calculating min/max over training set...
Done. rmin=-1.6025904539973852e-06, rmax=1.6123052658255991e-06, imin=-1.6147482377856819e-06, imax=1.5844537584687652e-06
Dataset statistical values set in the dataset (from all train-used frames).
frame-overlap count: 0

=== Data Check ===
ch : torch.Size([32, 16, 2048])  -> (B,T,F_in)
img: torch.Size([32, 16, 3, 224, 224])  -> (B,T,3,224,224) or (B,3,224,224) depending on your ds
y  : torch.Size([32, 4, 2048])  -> (B,P,F_out)
y stats | min: 0.1673, max: 0.8381
If scaling worked correctly, values should be within [0, 1].


# Train/Eval 함수

In [15]:
def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()

    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, img, y in loader:
        # Dataset이 이미 cuda 텐서를 반환하더라도 안전하게 유지
        ch = ch.to(device, non_blocking=True)
        img = img.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        yhat = model(ch, img)
        loss = F.mse_loss(yhat, y)
        
        loss.backward()

        if grad_clip is not None and grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        
        total_loss += loss.item()
        total_nmse += nmse_db(yhat.detach(), y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, img, y in loader:
        ch = ch.to(device, non_blocking=True)
        img = img.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        yhat = model(ch, img)
        loss = F.mse_loss(yhat, y)

        total_loss += loss.item()
        total_nmse += nmse_db(yhat, y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)


# 모델 생성 및 확인

In [16]:
backbone = multi_modal_lwm().to(device)
print("backbone element_length:", backbone.channel_embedding.element_length)
print("backbone d_model:", backbone.channel_embedding.d_model)
print("img dim check:", img.dim(), img.shape)

backbone element_length: 64
backbone d_model: 128
img dim check: 5 torch.Size([32, 16, 3, 224, 224])


In [17]:
img.shape

torch.Size([32, 16, 3, 224, 224])

In [18]:
ch.shape

torch.Size([32, 16, 2048])

In [19]:
ch, img, y = next(iter(train_loader))
F_in   = ch.shape[-1]       # 2048
P      = y.shape[1]         # pre_len (4)
F_out  = y.shape[-1]        # 2048

backbone = multi_modal_lwm().to(device)

model = FinetuneChannelPredictor(
    backbone=backbone,
    F_in=F_in,
    F_out=F_out,
    out_steps=P,             # ✅ 핵심
    pool="last",
    freeze_image=False,
    freeze_backbone=False,
    element_length=64,
    d_model=128
).to(device)

# Optimizer/Scheduler 설정

In [20]:
# requires_grad=True인 파라미터만 학습
trainable_params = [p for p in model.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in trainable_params))

optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

# (선택) cosine scheduler
epochs = 500
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


trainable params: 7167168


# 학습 루프 및 checkpoint 저장

In [21]:
import os
import time
import torch

best_val = float("inf")
best_val_nmse = None
best_epoch = None

t_train0 = time.time()

log_path = "lwm1_multi_modal_scene100_v2.txt"  # ✅ 파일명만 바꿔서 관리

with open(log_path, "w", encoding="utf-8") as f:
    # ✅ (선택) 실험 메타정보 헤더
    header = (
        "=== Experiment Info ===\n"
        f"model=LWM_multi_finetune\n"
        f"scenes=500\n"
        f"epochs={epochs}\n"
        f"F_in={F_in}, F_out={F_out}\n"
        f"start_time={time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        "=======================\n"
    )
    print(header)
    f.write(header + "\n")
    f.flush()

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        tr_loss, tr_nmse = train_one_epoch(model, train_loader, optimizer, device=device, grad_clip=1.0)
        va_loss, va_nmse = evaluate(model, val_loader, device=device)

        scheduler.step()

        dt = time.time() - t0
        line = (
            f"[{epoch:02d}/{epochs}] "
            f"train loss={tr_loss:.6f}, nmse(dB)={tr_nmse:.4f} | "
            f"val loss={va_loss:.6f}, nmse(dB)={va_nmse:.4f} | "
            f"{dt:.1f}s"
        )

        print(line)
        f.write(line + "\n")
        f.flush()

        if va_loss < best_val:
            best_val = va_loss
            best_val_nmse = va_nmse
            best_epoch = epoch

            ckpt_path = "best_LWM1_multi_model_finetune_scene100.pt"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "F_in": F_in,
                    "F_out": F_out,
                    "best_val_loss": best_val,
                    "best_val_nmse_db": best_val_nmse,
                },
                ckpt_path
            )

            save_line = (
                f"  ↳ saved {ckpt_path} | best@{best_epoch}: "
                f"val loss={best_val:.6f}, val nmse(dB)={best_val_nmse:.6f}"
            )
            print(save_line)
            f.write(save_line + "\n")
            f.flush()

total_sec = time.time() - t_train0
h = int(total_sec // 3600)
m = int((total_sec % 3600) // 60)
s = total_sec % 60

summary1 = "\n=== Training Summary ==="
summary2 = f"Total time: {total_sec:.1f}s ({h}h {m}m {s:.1f}s)"
summary3 = f"Best epoch: {best_epoch} | best val loss={best_val:.6f}, best val nmse(dB)={best_val_nmse:.6f}"

print(summary1); print(summary2); print(summary3)

with open(log_path, "a", encoding="utf-8") as f:
    f.write(summary1 + "\n")
    f.write(summary2 + "\n")
    f.write(summary3 + "\n")

print("Log saved to:", os.path.abspath(log_path))

=== Experiment Info ===
model=LWM_multi_finetune
scenes=500
epochs=500
F_in=2048, F_out=2048
start_time=2026-03-05 18:44:13

[01/500] train loss=0.583964, nmse(dB)=3.3528 | val loss=0.523338, nmse(dB)=3.1061 | 46.5s
  ↳ saved best_LWM1_multi_model_finetune_scene100.pt | best@1: val loss=0.523338, val nmse(dB)=3.106107
[02/500] train loss=0.536946, nmse(dB)=2.9880 | val loss=0.491620, nmse(dB)=2.8346 | 46.2s
  ↳ saved best_LWM1_multi_model_finetune_scene100.pt | best@2: val loss=0.491620, val nmse(dB)=2.834576
[03/500] train loss=0.507541, nmse(dB)=2.7461 | val loss=0.467550, nmse(dB)=2.6166 | 46.8s
  ↳ saved best_LWM1_multi_model_finetune_scene100.pt | best@3: val loss=0.467550, val nmse(dB)=2.616563
[04/500] train loss=0.487375, nmse(dB)=2.5273 | val loss=0.445685, nmse(dB)=2.4086 | 47.4s
  ↳ saved best_LWM1_multi_model_finetune_scene100.pt | best@4: val loss=0.445685, val nmse(dB)=2.408565
[05/500] train loss=0.462982, nmse(dB)=2.3534 | val loss=0.424910, nmse(dB)=2.2013 | 47.1s
  ↳ 

# 런타임 체크

In [22]:
ch, img, y = next(iter(train_loader))
print("y abs mean:", y.abs().mean().item())
print("y abs max :", y.abs().max().item())
print("y power   :", (y**2).mean().item())

with torch.no_grad():
    yhat = model(ch.to(device), img.to(device))
print("yhat abs mean:", yhat.abs().mean().item())
print("yhat abs max :", yhat.abs().max().item())
print("yhat power   :", (yhat**2).mean().item())


y abs mean: 0.4954802095890045
y abs max : 0.8380954265594482
y power   : 0.2689119279384613
yhat abs mean: 0.4986730217933655
yhat abs max : 0.5606470108032227
yhat power   : 0.24904990196228027


In [23]:
y[0:10,0:10]

tensor([[[0.6680, 0.6546, 0.6410,  ..., 0.7342, 0.7303, 0.7269],
         [0.3564, 0.3677, 0.3804,  ..., 0.4013, 0.4122, 0.4228],
         [0.7423, 0.7472, 0.7527,  ..., 0.7190, 0.7153, 0.7112],
         [0.2488, 0.2416, 0.2362,  ..., 0.5460, 0.5556, 0.5640]],

        [[0.6416, 0.6548, 0.6662,  ..., 0.3808, 0.3675, 0.3551],
         [0.3802, 0.3684, 0.3567,  ..., 0.7805, 0.7865, 0.7904],
         [0.7911, 0.7982, 0.8027,  ..., 0.4190, 0.4329, 0.4458],
         [0.2917, 0.2854, 0.2795,  ..., 0.4676, 0.4533, 0.4388]],

        [[0.6429, 0.6475, 0.6524,  ..., 0.3837, 0.3748, 0.3685],
         [0.6008, 0.5916, 0.5821,  ..., 0.4336, 0.4411, 0.4473],
         [0.4064, 0.4125, 0.4173,  ..., 0.6007, 0.5946, 0.5867],
         [0.4694, 0.4760, 0.4836,  ..., 0.4542, 0.4428, 0.4308]],

        ...,

        [[0.5586, 0.5688, 0.5797,  ..., 0.6582, 0.6710, 0.6817],
         [0.6315, 0.6318, 0.6311,  ..., 0.5701, 0.5622, 0.5518],
         [0.5919, 0.5841, 0.5780,  ..., 0.4748, 0.4650, 0.4556],
     

In [24]:
yhat[0:10:,0:10]

tensor([[[0.5522, 0.5531, 0.5511,  ..., 0.5137, 0.5111, 0.5079],
         [0.5510, 0.5503, 0.5529,  ..., 0.5079, 0.5054, 0.5043],
         [0.5474, 0.5461, 0.5442,  ..., 0.5109, 0.5093, 0.5069],
         [0.5326, 0.5333, 0.5333,  ..., 0.5201, 0.5220, 0.5184]],

        [[0.5520, 0.5525, 0.5515,  ..., 0.5135, 0.5107, 0.5077],
         [0.5508, 0.5504, 0.5528,  ..., 0.5081, 0.5056, 0.5044],
         [0.5471, 0.5460, 0.5444,  ..., 0.5110, 0.5093, 0.5066],
         [0.5325, 0.5336, 0.5333,  ..., 0.5206, 0.5210, 0.5191]],

        [[0.5523, 0.5535, 0.5516,  ..., 0.5136, 0.5115, 0.5075],
         [0.5512, 0.5492, 0.5534,  ..., 0.5077, 0.5047, 0.5035],
         [0.5468, 0.5450, 0.5447,  ..., 0.5109, 0.5104, 0.5066],
         [0.5326, 0.5329, 0.5340,  ..., 0.5203, 0.5192, 0.5193]],

        ...,

        [[0.5522, 0.5535, 0.5516,  ..., 0.5136, 0.5114, 0.5075],
         [0.5513, 0.5494, 0.5530,  ..., 0.5079, 0.5049, 0.5035],
         [0.5471, 0.5450, 0.5446,  ..., 0.5110, 0.5102, 0.5066],
     

In [25]:
ch, img, y = next(iter(train_loader))
y_persist = ch[:, -1, :]               # baseline
with torch.no_grad():
    yhat = model(ch.to(device), img.to(device))  # 멀티모달이면
# (채널-only면 model(ch.to(device))로)

def stats(name, t):
    print(name,
          "mean", t.mean().item(),
          "std",  t.std().item(),
          "min",  t.min().item(),
          "max",  t.max().item())

stats("y      ", y)
stats("persist", y_persist)
stats("yhat   ", yhat.cpu())

y       mean 0.4988704323768616 std 0.15317875146865845 min 0.16733872890472412 max 0.8380954265594482
persist mean 0.4988948106765747 std 0.15691356360912323 min 0.16733872890472412 max 0.8374385237693787
yhat    mean 0.4986732006072998 std 0.019367847591638565 min 0.443160742521286 max 0.5606470108032227


# 데이터 입력 형태

In [26]:
cam_files = sensor.files
print("=== dataset sizes ===")
print("N(comm_frames):", len(comm_frames))
print("N(cam_files)  :", len(cam_files))
print("N(min)        :", min(len(comm_frames), len(cam_files)))
print("past_len (T)  :", ds.past_len)
print("pre_len  (P)  :", getattr(ds, "pre_len", None))
print("len(ds)       :", len(ds))
print("len(train_ds) :", len(train_ds))
print("len(val_ds)   :", len(val_ds))
print("len(train_loader):", len(train_loader))
print("len(val_loader)  :", len(val_loader))

print("\n=== one batch shapes ===")
ch, img, y = next(iter(train_loader))

print("ch :", tuple(ch.shape),  " -> (B,T,F_in)")
print("img:", tuple(img.shape), " -> (B,T,3,224,224) or (B,3,224,224)")  # 너는 5D 허용
print("y  :", tuple(y.shape),   " -> (B,P,F_out)")

with torch.no_grad():
    yhat = model(ch.to(device), img.to(device))

print("yhat:", tuple(yhat.shape), " -> (B,P,F_out)")

B = yhat.shape[0]
P = yhat.shape[1] if yhat.ndim == 3 else 1
F = yhat.shape[-1]

print("this forward predicted samples:", B, "(=B)")
print("predicted steps per sample     :", P, "(=P)")
print("elements per step              :", F, "(=F_out)")

=== dataset sizes ===
N(comm_frames): 100
N(cam_files)  : 7012
N(min)        : 100
past_len (T)  : 16
pre_len  (P)  : 4
len(ds)       : 81
len(train_ds) : 45
len(val_ds)   : 15
len(train_loader): 2
len(val_loader)  : 1

=== one batch shapes ===
ch : (32, 16, 2048)  -> (B,T,F_in)
img: (32, 16, 3, 224, 224)  -> (B,T,3,224,224) or (B,3,224,224)
y  : (32, 4, 2048)  -> (B,P,F_out)
yhat: (32, 4, 2048)  -> (B,P,F_out)
this forward predicted samples: 32 (=B)
predicted steps per sample     : 4 (=P)
elements per step              : 2048 (=F_out)


In [ ]:
import torch

@torch.no_grad()
def nmse_db_batch(yhat, y, eps=1e-16):
    """
    yhat, y:
      - (B,F)   또는
      - (B,P,F)  (multi-step)
    return:
      - (B,)  각 샘플의 NMSE(dB)
    """
    if yhat.ndim == 2:
        # (B,F)
        num = torch.sum((yhat - y) ** 2, dim=1)
        den = torch.sum(y ** 2, dim=1).clamp_min(eps)
    elif yhat.ndim == 3:
        # (B,P,F) -> P,F 모두 합쳐서 샘플별 NMSE
        num = torch.sum((yhat - y) ** 2, dim=(1, 2))
        den = torch.sum(y ** 2, dim=(1, 2)).clamp_min(eps)
    else:
        raise ValueError(f"Unexpected shape: yhat={yhat.shape}, y={y.shape}")

    nmse = num / den
    return 10.0 * torch.log10(nmse.clamp_min(eps))  # (B,)


@torch.no_grad()
def compute_train_mean_vector(train_loader, device):
    """
    TRAIN target y의 feature-wise mean
    - y: (B,F)   -> return (F,)
    - y: (B,P,F) -> return (P,F)  (step별 평균 벡터)
    """
    s = None
    n = 0
    for ch, img, y in train_loader:
        y = y.to(device, non_blocking=True)
        if s is None:
            s = y.sum(dim=0)   # (F,) or (P,F)
        else:
            s += y.sum(dim=0)
        n += y.shape[0]
    mu = s / max(n, 1)
    return mu


@torch.no_grad()
def eval_model_and_baselines(model, loader, device, y_train_mean=None):
    model.eval()
    acc = {"model": [], "persist": [], "tmean": [], "trainmean": []}

    for ch, img, y in loader:
        ch  = ch.to(device, non_blocking=True)   # (B,T,F)
        img = img.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)    # (B,F) or (B,P,F)

        # 1) model
        yhat = model(ch, img)                    # (B,F) or (B,P,F)
        acc["model"].append(nmse_db_batch(yhat, y))

        # 2) persistence baseline
        if y.ndim == 2:
            y_persist = ch[:, -1, :]             # (B,F)
        else:
            P = y.shape[1]
            y_persist = ch[:, -1, :].unsqueeze(1).expand(-1, P, -1)  # (B,P,F)
        acc["persist"].append(nmse_db_batch(y_persist, y))

        # 3) time-mean baseline (past 평균)
        if y.ndim == 2:
            y_tmean = ch.mean(dim=1)             # (B,F)
        else:
            P = y.shape[1]
            y_tmean = ch.mean(dim=1).unsqueeze(1).expand(-1, P, -1)  # (B,P,F)
        acc["tmean"].append(nmse_db_batch(y_tmean, y))

        # 4) train-mean baseline
        if y_train_mean is not None:
            if y.ndim == 2:
                y_mu = y_train_mean.unsqueeze(0).expand_as(y)        # (B,F)
            else:
                y_mu = y_train_mean.unsqueeze(0).expand_as(y)        # (B,P,F) (mu가 (P,F)여야 함)
            acc["trainmean"].append(nmse_db_batch(y_mu, y))

    out = {}
    for k, lst in acc.items():
        if len(lst) == 0:
            continue
        out[k] = torch.cat(lst, dim=0).mean().item()
    return out


# ---- run ----
y_mu = compute_train_mean_vector(train_loader, device)
res_val = eval_model_and_baselines(model, val_loader, device, y_train_mean=y_mu)

print("=== VAL NMSE(dB) ===")
for k, v in res_val.items():
    print(f"{k:9s}: {v:.4f} dB")

if "model" in res_val and "persist" in res_val:
    print("Δ(model - persist) [dB improvement] =",
          (res_val["persist"] - res_val["model"]))

AttributeError: Caught AttributeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/dlghdbs200/anaconda3/envs/hoyun_312/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/dlghdbs200/anaconda3/envs/hoyun_312/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dlghdbs200/anaconda3/envs/hoyun_312/lib/python3.12/site-packages/torch/utils/data/dataset.py", line 413, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
            ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1435011/1281018576.py", line 36, in __getitem__
    img_k = preprocess_img(self.cam_files[k])   # (3,224,224)
            ^^^

: 